In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Values and the Bellman equation

## Where this session fits

> **The question we start from: how could we ever compute "the best behavior"?**

- **Position: Part A (value-based methods), session 1**
    - the tour gave the whole picture and named three things an agent can learn: a model, a value, a policy
    - this session takes the **value**: it turns "best behavior" into numbers we can write equations for
    - the next session solves those equations exactly when the environment's rules are known
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. say what a **value** is and what the **value function** $\statevalfun_\pi$ is, and read a value table
    - a value is one number for a state
    - the value function is the rule giving it for every state
2. write the Bellman equation, in its averaging version and its optimality (max) version, and say what a backup is
3. check the optimality equation by hand at a square of Frozen Lake
4. say why values give a policy (a one-step lookahead that uses the model)
5. say why the Bellman equation is not yet a method: the value is on both sides


# What is a value?

**Why this section?** The tour named three things an agent can learn. We start with the *value*: one number per state that says how good it is to be there.

Recall the return $\rlret_\tt = \sum_{k \geq 0} \disc^k \, \rewseq_{\tt+k+1}$ (the discounted rewards from time $\tt$ on). Value-based methods revolve around the idea of
- creating a **state value function** $\statevalfun_\pi(\state)$ 
- to assign a value to each state $\state$ in the set of states $\States$
$$
\statevalfun_{\pi}: \States \to \Reals
$$
(one real number, in $\Reals$, per state)

$\statevalfun_\pi(\state)$ is the expectation 
$$
\Exp{\pi}\left[ \rlret_\tt \mid \stateseq_\tt = \state \right]
$$
the expected return from state $\state$ when following $\pi$

- if nothing is random (no chance in the environment or in $\pi$), the expectation is just the return of the one path that results
- the **value** is that one number
- the **value function** is the whole assignment, state by state (a **table** when the states are few)

**Frozen Lake in this session**

<img src="images/Frozen_Lake_env.png" width=18%>

- the grid has $16$ squares, numbered $0$ to $15$ row by row from the top left
    - **S** is square $0$, **G** is square $15$
    - **F** are frozen squares, **H** are holes
- four actions: left, down, right, up
- **non-slippery** here: every move goes where intended
    - a simplification of the real, slippery lake of the tour
    - so the numbers differ from the tour's slippery model panel
- reward: $+1$ when a move enters **G**, $0$ otherwise
- the episode ends at **G** or at a hole **H**, and the discount is $\disc = 0.9$
- terminal squares have value $0$: the $+1$ is earned by the move into **G**, not by being there

**The model of this lake**

- the **model** is the environment's rule for what happens after an action: $\transpm(\state', \rew \mid \state, \act)$
    - the chance that taking action $\act$ in square $\state$ leads to square $\state'$ with reward $\rew$
    - the book's joint form, used throughout the series
- on this non-slippery lake one pair $(\state', \rew)$ has probability $1$ and every other pair $0$
- a move into the outer wall leaves the agent where it is
- here *model-based* means the model is **known** and used for planning (next session)
- *model-free* means we only get **samples** by interacting with the environment (the session after)
    - this is narrower than the common use of "model-based", which also includes *learning* the model
    - this series does not cover learning a model


We usually represent (and implement) $\statevalfun_\pi(\state)$ as a table.
- for the case of finite number of states
- each cell is a state and holds the expected return from it
    - colour shows magnitude: pale is $0$, deeper is higher
- on Frozen Lake the single $+1$ reward keeps every value in $[0, 1]$

For Frozen Lake a table with every value set to $0$ (the starting point of the methods that follow) looks like:

<br>
<table>
    <center><strong>Value function (uninitialized) as a table</strong></center>
<tr>
    <td><img src="images/Frozen_Lake_env.png" width=50%></td>
    <td><img src="images/value_table_zero.png" width=50%></td>
</tr>
</table>

# Interlude: the Bellman recursion and backups

**Why this section?** The value of a state is tied to the values of the states that follow it. That relation, the Bellman equation, drives every method in this series. We meet it once, carefully, here.

## **Definition: a backup**

> **A backup is an update of a value from the values of what follows it.**
> Information flows from successor to predecessor.

- later sessions qualify the word only where it matters
    - a **full** backup uses every successor, a **sampled** one uses one (A3)
- repeated backups carry the reward back along the route, one step at a time
    - from states near the reward to states far from it
- every value-based method in this series is built on backups

**Heads-up: one value, written two ways, then estimated by sampling**

> **The value is an average over all possible trajectories.**

- **trajectory form:** the average of the return $\rlret_\tt$ over the trajectories $\traj$ from a state, weighted by their chance
    - the recursion on the next slide describes one episode of it
- **one-step form:** the Bellman equation, the same average written with no trajectories, only sums over actions and next states
    - both forms are exact
    - a short step connects them (a slide of its own)
- **sampled estimate:** use the returns of sampled trajectories in place of the average (A3)
    - an approximation that approaches the average as the samples grow

**The return satisfies a recursion** (the trajectory form)

$$
\rlret_\tt = \rewseq_{\tt+1} + \disc\,\rlret_{\tt+1}
$$

- $\rlret_\tt$ is the return from step $\tt$ **of one trajectory $\traj$** (one episode)
    - we do not write $\traj$ in the symbol
    - the trajectory at hand is understood
    - it differs from trajectory to trajectory, because chance changes the path
- along that episode, the return from step $\tt$ is the next reward plus the discounted return from step $\tt+1$
- the return now is the next reward plus the return one step later
- **one sampled path** gives the return recursion above
- **averaging over everything that can happen** gives the **Bellman equation**
    - the expected return from a state is the sum of two terms
        - the expected first reward
        - $\disc$ times the expected value of the next state
    - sampling versus averaging comes back in later sessions

**From trajectories to a one-step equation**

> **The link between the two forms: average the recursion over the first step only.**

$$
\statevalfun_\pi(\state) = \Exp{\pi}\left[ \rlret_\tt \mid \stateseq_\tt = \state \right] = \Exp{\pi}\left[ \rewseq_{\tt+1} + \disc\,\rlret_{\tt+1} \mid \stateseq_\tt = \state \right]
$$

- the first equals sign is the definition: an average over whole trajectories $\traj$ that start in $\state$
- the second inserts the return recursion
- only the first step needs to be written out as a sum
    - the action is drawn from $\pi(\act \mid \state)$, which gives $\sum_{\act} \pi(\act \mid \state)$
    - the next state and reward are drawn from $\transpm(\state', \rew \mid \state, \act)$, which gives $\sum_{\state', \rew}$


**What is left of the trajectory is one number**

- after the first step the future is $\rlret_{\tt+1}$, the return from the state we landed in, $\state'$
- its average is, by definition, a value: $\Exp{\pi}\left[ \rlret_{\tt+1} \mid \stateseq_{\tt+1} = \state' \right] = \statevalfun_\pi(\state')$
- this uses the **Markov property**: the future depends on the state we landed in, not on the path that led there
- so the whole rest of the trajectory is replaced by one number, $\statevalfun_\pi(\state')$
- the result has no trajectories in it, only $\pi$, $\transpm$ and $\statevalfun_\pi$ (next slide)


The Bellman equation says that the value of a state is an average over what can happen next (the one-step form)
$$
\statevalfun_\pi(\state) = \sum_{\act} \pi(\act \mid \state) \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun_\pi(\state')\bigr)
$$

- **why two sums?** two things can be random
    - the policy may be random: $\pi(\act \mid \state)$ weights the actions
    - the environment may be random: the same action in the same state can lead to different next states $\state'$ (and rewards $\rew$)
        - they are weighted by $\transpm(\state', \rew \mid \state, \act)$
- example: on the real, *slippery* lake a move goes the intended way one time in three
    - otherwise it slides to a side
    - so up to three successor squares each have weight $1/3$
    - the inner sum is a true average
- on today's non-slippery lake every move has one successor (weight $1$), so the inner sum has a single term


**Two versions of the equation: average over actions, or choose the best**

- **for a given policy $\pi$** (the equation above): the policy decides, so the value *averages* over the actions it might take
- **for the best behavior** (the *optimality* form): the best policy picks the best action in every state, so the average is replaced by a **max**

$$
\statevalfun_*(\state) = \max{\act} \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun_*(\state')\bigr)
$$

- $\statevalfun_\pi$: how good is this state *if we follow $\pi$*
- $\statevalfun_*$: how good is it *under the best behavior*
- both are one equation per state, with the value of the successor states on the right-hand side
- the optimality version is what "computing the best behavior" will mean: find the $\statevalfun_*$ that satisfies it


**The forms of a value, side by side**

> **Two exact forms. Sampling either one gives an approximation.**

| form | exact or approximate? | what it needs | where it is used |
|:--|:--|:--|:--|
| **trajectory form:** the average of the return over all trajectories from a state | exact | the model, to list every trajectory (impractical) | the definition of the value |
| **one-step form:** the Bellman equation, with no trajectories | exact | the model, to average over the successors | A2 (dynamic programming) |
| **sampled trajectory form:** the return of one sampled episode | approximate | episodes only | A3 (Monte Carlo) |
| **sampled one-step form:** one sampled reward and next state, plus the current estimate | approximate | transitions only | A3 (TD), A4 (Q-learning, SARSA), A5 (deep Q-networks) |

- later sessions flag which form they are using


**Where this equation comes back**

- the one-step form is the template for the whole of Part A
- each later session modifies it
    - **A2, dynamic programming:** the equation becomes an *update rule*, applied to every state again and again
    - **A3, sampling:** the average over $\transpm$ is replaced by *one sampled* reward and next state
    - **A4, control:** the same equation written for action values $\actvalfun$, with the max version for the best behavior
    - **A5, deep Q-networks:** the right-hand side becomes the *target* a network is trained to match
- so it is worth being sure of this one: the later equations are this one with a part replaced


## From values to a policy

**Why this section?** The point of values is acting. A value table gives a policy by looking one step ahead.

- the value of a state says how good it is
- to *act*, we need to choose an action


Given the optimal value function $\statevalfun_*$, an optimal deterministic policy looks one step ahead

$$
\pi_*(\state) = \argmax{\act} \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun_*(\state')\bigr)
$$

- From state $\state$
- try each action $\act$
- use the model $\transpm$ to weigh each possible next state $\state'$
- choose the action with the best reward plus discounted value of what follows

Note: the $\argmax{}$ results in a *deterministic* policy: one action in each state
- we write $\pi(\state)$ for that action
- as a distribution, $\pi(\act \mid \state)$ is $1$ for it and $0$ for every other action

Without the model this lookahead is impossible. Action values (session A4, on control) make the last step model-free.


Here is the answer we are after: the optimal values and policy (non-slippery, discount $0.9$), as computed by the method of the next session (the picture's title mentions its sweeps)
- $\statevalfun_*(\state)$ 
- and an optimal policy $\pi_*(\state)$, shown as arrows

<table>
    <center><strong>Value function (converged) as a table</strong></center>
<tr>
    <td><img src="images/value_iteration_final.png" width=75%></td>
</tr>
</table>

Where two actions tie (squares $0$ and $9$), one arrow is shown.


## Checking the optimality equation by hand

**Why this section?** The equation should hold for the optimal values. Check it at two squares, using the optimal values of the earlier picture (non-slippery, $\disc = 0.9$): $\statevalfun_*(14) = 1$, $\statevalfun_*(13) = 0.9$, $\statevalfun_*(10) = 0.9$, $\statevalfun_*(9) = 0.81$, $\statevalfun_*(6) = 0.81$.

- square $10$: its neighbors are $9$ (left), $6$ (up), $11$ (right, a hole) and $14$ (down)
- $\statevalfun_*(9) = \statevalfun_*(6) = 0.81$, $\statevalfun_*(14) = 1$, a hole has value $0$
    - down: $0 + 0.9 \times 1 = 0.9$
    - left: $0 + 0.9 \times 0.81 = 0.729$
    - up: also $0.729$
    - right (into the hole): $0 + 0.9 \times 0 = 0$
    - the max is $0.9$, which equals $\statevalfun_*(10)$: the equation holds


**The check at square 6, and at the start**

- square $6$: neighbors $2$ (up, $0.729$), $10$ (down, $0.9$), and holes $5$, $7$
    - down: $0.9 \times 0.9 = 0.81$
    - up: $0.9 \times 0.729 \approx 0.656$
    - into a hole: $0$
    - the max is $0.81 = \statevalfun_*(6)$
- each such backup moves value one square back from the goal, shrunk by $\disc$


## The limitation: the equation is not yet an answer

**Why this section?** We can write down what the best behavior must satisfy, and check it when given the answer, but we have not computed it.

- the Bellman equation has the value on **both sides**: $\statevalfun(\state)$ depends on the values of the successors, which depend on theirs
- so we cannot just evaluate it
- the idea: **iterate**
    - start from a table of zeros (the table of zeros from "What is a value?")
    - apply the backup to every square, again and again
        - each pass carries the reward one square further back
    - the sequence of tables $\statevalfun_0, \statevalfun_1, \ldots$ should approach $\statevalfun_*$
- how well this works, and for which equation (average or max), is the next session


# What we established, and what it sets up

- **Established**
    - the state value $\statevalfun_\pi(\state)$ is the expected return from a state
    - on a small problem it is a table
    - the **Bellman equation** relates a state's value to its successors' values
    - a **backup** carries value from successors to predecessors
    - two forms of one idea: the return recursion along one episode (the trajectory form) and the Bellman equation with no trajectories (the one-step form)
        - averaging the recursion over the first step links them
    - two versions of the equation
        - an average over the policy's actions, for $\statevalfun_\pi$
        - a max for the best behavior, $\statevalfun_*$ (checked by hand)
    - a value table gives a policy by a one-step lookahead that uses the model
- **Sets up (the limitation)**
    - the equation has the value on both sides, so it is not yet a method
    - next session solves it by iterating (dynamic programming), when the model is known
    - then: what if the model is not known?
